# 📐 Taller - Visualización y Cálculo de Proyección 3D

**Objetivo:** Representar puntos en 3D con coordenadas homogéneas, implementar matrices de proyección ortogonal y perspectiva, y visualizar gráficamente el efecto de cada transformación.

**Herramientas:** `numpy`, `matplotlib`, `ipywidgets`

**Figura usada:** Casa 3D — elegida porque combina superficies planas (paredes) y una estructura angular (techo), haciendo muy visible la diferencia entre tipos de proyección.

---
**Contenido del notebook:**
1. Instalación y configuración
2. Fundamento teórico: coordenadas homogéneas
3. Definición de la figura 3D (casa)
4. Proyección Ortogonal
5. Proyección Perspectiva
6. Comparación visual lado a lado
7. Efecto de la distancia focal (interactivo con sliders)
8. Visualización 3D del volumen de proyección

## 📦 Sección 1 — Instalación y configuración

In [ ]:
!pip install numpy matplotlib ipywidgets --quiet

# Habilitamos los widgets interactivos en Colab
from google.colab import output
output.enable_custom_widget_manager()

print('✅ Dependencias listas.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import ipywidgets as widgets
from IPython.display import display

# Estilo visual consistente en todas las gráficas
plt.rcParams.update({
    'figure.facecolor' : '#0f1117',
    'axes.facecolor'   : '#1a1d27',
    'axes.edgecolor'   : '#3a3f55',
    'axes.labelcolor'  : '#c8cce0',
    'xtick.color'      : '#6a6f8a',
    'ytick.color'      : '#6a6f8a',
    'text.color'       : '#c8cce0',
    'grid.color'       : '#2a2f45',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'font.family'      : 'monospace',
})

print('✅ Importaciones listas.')

## 📖 Sección 2 — Fundamento teórico: Coordenadas Homogéneas

Las **coordenadas homogéneas** añaden una dimensión extra $w$ a los vectores, permitiendo representar traslaciones como multiplicaciones de matrices. Un punto 3D $(x, y, z)$ se representa como:

$$\mathbf{p} = \begin{pmatrix} x \\ y \\ z \\ w \end{pmatrix} \quad \text{donde } w = 1$$

Para convertir de vuelta a coordenadas 3D, se divide por $w$:

$$\text{3D} = \left(\frac{x}{w},\ \frac{y}{w},\ \frac{z}{w}\right)$$

### Matriz de Proyección Ortogonal
Elimina la coordenada Z sin considerar la distancia. Los objetos lejanos y cercanos se ven igual de grandes:

$$P_{\text{ortogonal}} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix}$$

### Matriz de Proyección Perspectiva
Divide X e Y por Z (escalado por la distancia focal $d$), simulando cómo el ojo humano percibe la profundidad. Los objetos lejanos se ven más pequeños:

$$P_{\text{perspectiva}} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 1/d & 0 \end{pmatrix}$$

## 🏠 Sección 3 — Definición de la figura 3D: Casa

In [ ]:
# ── Vértices de la casa ──────────────────────────────────────
# La casa se define como un conjunto de puntos 3D (x, y, z).
# Cada columna es un vértice. Los uniremos con aristas para
# formar la estructura completa.
#
#        6
#       /|\
#      / | \
#     4--5--7      <- techo (y=2.5)
#     |     |
#     |     |
#     0-----1      <- piso (y=0)
#     |     |
#     3-----2

vertices = np.array([
    # Base inferior (piso)
    [-1, 0, -1],   # 0 - frente izquierdo
    [ 1, 0, -1],   # 1 - frente derecho
    [ 1, 0,  1],   # 2 - atrás derecho
    [-1, 0,  1],   # 3 - atrás izquierdo
    # Base superior (techo plano)
    [-1, 1.5, -1], # 4 - frente izquierdo alto
    [ 1, 1.5, -1], # 5 - frente derecho alto
    [ 1, 1.5,  1], # 6 - atrás derecho alto
    [-1, 1.5,  1], # 7 - atrás izquierdo alto
    # Punta del techo
    [ 0, 2.5, -1], # 8 - cumbrera frente
    [ 0, 2.5,  1], # 9 - cumbrera atrás
]).T  # Transponemos para que cada columna sea un vértice (3 x N)

# ── Aristas de la casa ───────────────────────────────────────
# Cada tupla (i, j) conecta el vértice i con el vértice j
aristas = [
    # Paredes base
    (0,1), (1,2), (2,3), (3,0),
    # Paredes techo
    (4,5), (5,6), (6,7), (7,4),
    # Columnas verticales
    (0,4), (1,5), (2,6), (3,7),
    # Techo - triángulo frente
    (4,8), (5,8),
    # Techo - triángulo atrás
    (7,9), (6,9),
    # Cumbrera
    (8,9),
]

print(f'🏠 Casa definida con {vertices.shape[1]} vértices y {len(aristas)} aristas.')
print(f'   Shape de vertices: {vertices.shape}  (3 coordenadas x {vertices.shape[1]} puntos)')

In [ ]:
# ── Visualización 3D de la casa original ─────────────────────
fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#1a1d27')

# Dibujamos cada arista
for i, j in aristas:
    ax.plot(
        [vertices[0,i], vertices[0,j]],
        [vertices[2,i], vertices[2,j]],  # Z como profundidad
        [vertices[1,i], vertices[1,j]],  # Y como altura
        color='#4fc3f7', linewidth=1.8, alpha=0.9
    )

# Dibujamos los vértices
ax.scatter(vertices[0], vertices[2], vertices[1],
           color='#ffcc02', s=40, zorder=5)

# Etiquetas de vértices
for idx in range(vertices.shape[1]):
    ax.text(vertices[0,idx], vertices[2,idx], vertices[1,idx]+0.08,
            str(idx), color='#ffcc02', fontsize=8)

ax.set_xlabel('X'); ax.set_ylabel('Z'); ax.set_zlabel('Y')
ax.set_title('🏠 Casa 3D — Figura original', pad=15, fontsize=13)
ax.tick_params(colors='#6a6f8a')
plt.tight_layout()
plt.savefig('casa_3d_original.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Vista 3D de la casa generada.')

## 📦 Sección 4 — Funciones de Proyección

In [ ]:
# ── Función: Proyección Ortogonal ────────────────────────────
def proyectar_ortogonal(puntos):
    """
    Aplica proyección ortogonal a un conjunto de puntos 3D.
    Simplemente descarta la coordenada Z — no hay perspectiva.
    Todos los puntos se proyectan con el mismo tamaño independientemente
    de su distancia a la cámara.

    Matriz:
        P = [[1, 0, 0, 0],
             [0, 1, 0, 0],
             [0, 0, 0, 0],
             [0, 0, 0, 1]]

    Args:
        puntos: array (3, N) con N puntos 3D
    Returns:
        array (2, N) con los puntos proyectados en 2D
    """
    # Matriz de proyección ortogonal en coordenadas homogéneas
    P = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 1]
    ])

    # Convertimos a coordenadas homogéneas: añadimos fila de unos
    n = puntos.shape[1]
    puntos_hom = np.vstack((puntos, np.ones((1, n))))  # shape: (4, N)

    # Aplicamos la matriz de proyección
    proy = P @ puntos_hom  # shape: (4, N)

    # Normalizamos dividiendo por w (última fila)
    # En proyección ortogonal w=1 siempre, pero lo hacemos
    # por consistencia con el método perspectiva
    proy = proy / proy[-1, :]

    # Retornamos solo X e Y (descartamos Z=0 y w=1)
    return proy[:2, :]


# ── Función: Proyección Perspectiva ──────────────────────────
def proyectar_perspectiva(puntos, d=1.0):
    """
    Aplica proyección perspectiva a un conjunto de puntos 3D.
    Los objetos más lejanos (mayor Z) se ven más pequeños,
    simulando la visión humana y las lentes de cámara.

    Matriz:
        P = [[1, 0,   0,   0],
             [0, 1,   0,   0],
             [0, 0,   1,   0],
             [0, 0, 1/d,   0]]

    El truco está en que w = Z/d, y al dividir por w obtenemos:
        x' = x * d/Z
        y' = y * d/Z
    Esto hace que puntos con Z grande (lejos) se achiquen.

    Args:
        puntos: array (3, N) con N puntos 3D
        d: distancia focal. Mayor d = menos distorsión perspectiva.
           Menor d = perspectiva más pronunciada (gran angular).
    Returns:
        array (2, N) con los puntos proyectados en 2D
    """
    P = np.array([
        [1, 0,   0, 0],
        [0, 1,   0, 0],
        [0, 0,   1, 0],
        [0, 0, 1/d, 0]
    ])

    # Coordenadas homogéneas
    n = puntos.shape[1]
    puntos_hom = np.vstack((puntos, np.ones((1, n))))  # (4, N)

    # Aplicamos la proyección
    proy = P @ puntos_hom  # (4, N)

    # CLAVE: w = Z/d (varía por punto según su profundidad)
    # Al dividir, los puntos lejanos se comprimen hacia el centro
    proy = proy / proy[-1, :]  # divide cada columna por su w

    return proy[:2, :]


# ── Función auxiliar: dibujar casa proyectada ────────────────
def dibujar_proyeccion(ax, puntos_2d, aristas, color, titulo, subtitulo=''):
    """
    Dibuja las aristas y vértices de la casa proyectada en 2D.
    """
    for i, j in aristas:
        ax.plot(
            [puntos_2d[0,i], puntos_2d[0,j]],
            [puntos_2d[1,i], puntos_2d[1,j]],
            color=color, linewidth=1.8, alpha=0.9
        )
    ax.scatter(puntos_2d[0], puntos_2d[1], color='#ffcc02', s=25, zorder=5)
    ax.set_title(titulo, fontsize=12, pad=10)
    if subtitulo:
        ax.set_xlabel(subtitulo, fontsize=9, labelpad=8)
    ax.set_aspect('equal')
    ax.grid(True)
    ax.axhline(0, color='#3a3f55', linewidth=0.8)
    ax.axvline(0, color='#3a3f55', linewidth=0.8)


print('✅ Funciones de proyección definidas.')

## 🔲 Sección 5 — Proyección Ortogonal

In [ ]:
# Aplicamos la proyección ortogonal a todos los vértices de la casa
proy_ortogonal = proyectar_ortogonal(vertices)

print('📐 Proyección Ortogonal')
print('=' * 40)
print(f'   Vértices originales (3D)  : shape {vertices.shape}')
print(f'   Vértices proyectados (2D) : shape {proy_ortogonal.shape}')
print()
print('   Vértice  |  X original  |  Y original  |  Z (descartada)  |  X proy  |  Y proy')
print('   ' + '-'*80)
for i in range(vertices.shape[1]):
    print(f'      {i}     |   {vertices[0,i]:+.2f}       |   {vertices[1,i]:+.2f}       |   {vertices[2,i]:+.2f}          |  {proy_ortogonal[0,i]:+.2f}   |  {proy_ortogonal[1,i]:+.2f}')

# Visualización
fig, ax = plt.subplots(figsize=(6, 6))
dibujar_proyeccion(
    ax, proy_ortogonal, aristas,
    color='#4fc3f7',
    titulo='🔲 Proyección Ortogonal',
    subtitulo='La coordenada Z es descartada. Sin efecto de profundidad.'
)
plt.tight_layout()
plt.savefig('proyeccion_ortogonal.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔺 Sección 6 — Proyección Perspectiva

In [ ]:
# Desplazamos la casa en Z y la rotamos para que la perspectiva
# sea claramente visible — sin esto la figura queda centrada en Z≈0
# y la diferencia entre frente y fondo es imperceptible

D = 2.0

# Rotación en Y de 25° para ver la casa de costado
ry = np.radians(25)
Ry = np.array([
    [ np.cos(ry), 0, np.sin(ry)],
    [0,           1, 0         ],
    [-np.sin(ry), 0, np.cos(ry)]
])

# Rotación en X de 15° para ver el techo
rx = np.radians(15)
Rx = np.array([
    [1,       0,        0      ],
    [0, np.cos(rx), -np.sin(rx)],
    [0, np.sin(rx),  np.cos(rx)]
])

# Aplicamos rotación y desplazamos en Z dinámicamente
verts_rot = Ry @ Rx @ vertices
z_min = verts_rot[2, :].min()
verts_rot[2, :] += (-z_min + 3.0)   # garantiza Z > 0 en todos los vértices

# Proyectamos
proy_ortogonal_s6   = proyectar_ortogonal(verts_rot)
proy_perspectiva_s6 = proyectar_perspectiva(verts_rot, d=D)

# Tabla numérica
print(f'📐 Proyección Perspectiva  (d = {D})')
print('=' * 40)
print(f'   Fórmula: x\' = x·d/Z,  y\' = y·d/Z')
print()
print('   Vértice  |  X rot  |  Y rot  |  Z (profundidad)  |  X proy  |  Y proy')
print('   ' + '-'*75)
for i in range(verts_rot.shape[1]):
    print(f'      {i}     |  {verts_rot[0,i]:+.2f}   |  {verts_rot[1,i]:+.2f}   |   {verts_rot[2,i]:+.2f}          |  {proy_perspectiva_s6[0,i]:+.4f}  |  {proy_perspectiva_s6[1,i]:+.4f}')

# Visualización lado a lado para que la diferencia sea obvia
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

dibujar_proyeccion(
    axes[0], proy_ortogonal_s6, aristas,
    color='#c0c0c0',
    titulo='🔲 Ortogonal (referencia)',
    subtitulo='Aristas paralelas, sin profundidad'
)

dibujar_proyeccion(
    axes[1], proy_perspectiva_s6, aristas,
    color='#81c784',
    titulo=f'🔺 Proyección Perspectiva  (d={D})',
    subtitulo='Las aristas convergen hacia el punto de fuga'
)

# Anotamos el punto de fuga aproximado en la vista perspectiva
axes[1].axhline(0, color='#ffcc02', linewidth=0.6, linestyle=':', alpha=0.5)
axes[1].axvline(0, color='#ffcc02', linewidth=0.6, linestyle=':', alpha=0.5)
axes[1].annotate('Punto de fuga ≈ (0,0)', xy=(0,0), fontsize=8,
                 color='#ffcc02', xytext=(0.05, 0.05),
                 textcoords='axes fraction')

fig.suptitle('🔺 Proyección Perspectiva — Vista con rotación para mayor claridad',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('proyeccion_perspectiva.png', dpi=120, bbox_inches='tight')
plt.show()

## ⚖️ Sección 7 — Comparación lado a lado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Panel izquierdo: Ortogonal ────────────────────────────────
# Tres casas a distintas profundidades, todas del mismo tamaño
desplazamientos_z = [3, 6, 10]   # distancias a la cámara
colores_casas     = ['#4fc3f7', '#ffcc02', '#ff7043']
etiquetas         = ['Cerca  (Z=3)', 'Media  (Z=6)', 'Lejos  (Z=10)']

for z_offset, color, etiqueta in zip(desplazamientos_z, colores_casas, etiquetas):
    # Desplazamos la casa en Z
    v_desplazado = vertices.copy()
    v_desplazado[2, :] += z_offset

    pp = proyectar_ortogonal(v_desplazado)

    # Desplazamos horizontalmente para que no se encimen
    offset_x = (z_offset - 6) * 0.4
    for i, j in aristas:
        axes[0].plot(
            [pp[0,i] + offset_x, pp[0,j] + offset_x],
            [pp[1,i],            pp[1,j]],
            color=color, linewidth=1.8, alpha=0.9, label=etiqueta if i==0 else ''
        )

axes[0].set_title('🔲 Ortogonal\nLas tres casas son idénticas sin importar la distancia',
                  fontsize=11, pad=10)
axes[0].set_aspect('equal')
axes[0].grid(True)
axes[0].axhline(0, color='#3a3f55', linewidth=0.8)
axes[0].axvline(0, color='#3a3f55', linewidth=0.8)
axes[0].legend(loc='upper right', fontsize=9)

# ── Panel derecho: Perspectiva ────────────────────────────────
# Las mismas tres casas pero con perspectiva — las lejanas se achican
D = 2.0

for z_offset, color, etiqueta in zip(desplazamientos_z, colores_casas, etiquetas):
    v_desplazado = vertices.copy()
    v_desplazado[2, :] += z_offset

    pp = proyectar_perspectiva(v_desplazado, d=D)

    for i, j in aristas:
        axes[1].plot(
            [pp[0,i], pp[0,j]],
            [pp[1,i], pp[1,j]],
            color=color, linewidth=1.8, alpha=0.9, label=etiqueta if i==0 else ''
        )

axes[1].set_title(f'🔺 Perspectiva (d={D})\nLas casas lejanas se ven más pequeñas',
                  fontsize=11, pad=10)
axes[1].set_aspect('equal')
axes[1].grid(True)
axes[1].axhline(0, color='#3a3f55', linewidth=0.8)
axes[1].axvline(0, color='#3a3f55', linewidth=0.8)
axes[1].legend(loc='upper right', fontsize=9)

fig.suptitle('⚖️ Ortogonal vs Perspectiva — 3 casas a distintas profundidades',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('comparacion_profundidad.png', dpi=120, bbox_inches='tight')
plt.show()

print('📝 Observa:')
print('   Ortogonal → las 3 casas tienen exactamente el mismo tamaño')
print('   Perspectiva → la casa azul (Z=3) es grande, la roja (Z=10) es pequeña')
print('   Esa reducción de tamaño con la distancia es el efecto perspectiva.')

## 🎛️ Sección 8 — Efecto de la Distancia Focal (Interactivo)

In [ ]:
# Widget interactivo con ipywidgets.
# Al mover el slider, la proyección perspectiva se recalcula
# y la gráfica se actualiza en tiempo real.

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# Configuramos el backend interactivo
%matplotlib inline

def actualizar_proyeccion(d, rotacion_x, rotacion_y):
    """
    Recalcula y grafica la proyección perspectiva con los parámetros
    dados por los sliders. También permite rotar la figura en 3D
    antes de proyectarla para ver diferentes ángulos.
    """

    # ── Matriz de rotación en X ───────────────────────────────
    rx = np.radians(rotacion_x)
    Rx = np.array([
        [1,       0,        0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx),  np.cos(rx)]
    ])

    # ── Matriz de rotación en Y ───────────────────────────────
    ry = np.radians(rotacion_y)
    Ry = np.array([
        [ np.cos(ry), 0, np.sin(ry)],
        [0,           1, 0         ],
        [-np.sin(ry), 0, np.cos(ry)]
    ])

    # Aplicamos rotación a los vértices originales
    verts_rot = Ry @ Rx @ vertices

    # Desplazamos la figura en Z para que quede frente a la cámara
    z_min = verts_rot[2, :].min()
    margen = 3.0   # distancia mínima a la cámara
    verts_rot[2, :] += (-z_min + margen)

    # Calculamos ambas proyecciones
    pp_ortogonal    = proyectar_ortogonal(verts_rot)
    pp_perspectiva  = proyectar_perspectiva(verts_rot, d=d)

    # ── Figura con dos paneles ────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    dibujar_proyeccion(
        axes[0], pp_ortogonal, aristas,
        color='#c0c0c0',
        titulo='🔲 Ortogonal (referencia)',
        subtitulo='Sin efecto de profundidad'
    )

    dibujar_proyeccion(
        axes[1], pp_perspectiva, aristas,
        color='#81c784',
        titulo=f'🔺 Perspectiva — d = {d:.2f}',
        subtitulo=f'd={d:.2f} | rot_x={rotacion_x}° | rot_y={rotacion_y}°'
    )

    # Anotación del tipo de lente según el valor de d
    if d < 0.8:
        tipo = '📷 Gran angular (distorsión alta)'
    elif d < 2.0:
        tipo = '📷 Lente normal'
    elif d < 4.0:
        tipo = '🔭 Teleobjetivo leve'
    else:
        tipo = '🔭 Teleobjetivo (converge a ortogonal)'

    fig.suptitle(f'{tipo}  —  Distancia focal d = {d:.2f}',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# ── Definición de los sliders ────────────────────────────────
slider_d = widgets.FloatSlider(
    value=2.0, min=0.2, max=8.0, step=0.1,
    description='Focal d:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False,   # Solo actualiza al soltar el slider
    readout_format='.1f'
)

slider_rx = widgets.IntSlider(
    value=15, min=-60, max=60, step=5,
    description='Rot X:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False
)

slider_ry = widgets.IntSlider(
    value=25, min=-90, max=90, step=5,
    description='Rot Y:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False
)

# Conectamos los sliders con la función de actualización
out = widgets.interactive_output(
    actualizar_proyeccion,
    {'d': slider_d, 'rotacion_x': slider_rx, 'rotacion_y': slider_ry}
)

# Mostramos los controles y la salida
controles = widgets.VBox([
    widgets.HTML('<b style="color:#c8cce0">🎛️ Controles de proyección:</b>'),
    slider_d,
    slider_rx,
    slider_ry
])

display(controles, out)

## 🌐 Sección 9 — Visualización del Volumen de Proyección

In [ ]:
# Visualizamos el frustum (pirámide de visión) de la proyección perspectiva
# comparado con el volumen ortogonal (caja rectangular).
# Esto ayuda a entender geométricamente por qué la perspectiva
# produce el efecto de profundidad.

fig = plt.figure(figsize=(13, 5))

# ── Volumen Ortogonal (caja) ──────────────────────────────────
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_facecolor('#1a1d27')

# Plano near (cercano)
near = np.array([[-1,-1,0],[1,-1,0],[1,1,0],[-1,1,0]])
# Plano far (lejano)
far  = np.array([[-1,-1,3],[1,-1,3],[1,1,3],[-1,1,3]])

# Caras del volumen ortogonal
caras_orto = [
    [near[0],near[1],near[2],near[3]],
    [far[0], far[1], far[2], far[3]],
    [near[0],near[1],far[1], far[0]],
    [near[2],near[3],far[3], far[2]],
]
poly_orto = Poly3DCollection(caras_orto, alpha=0.12, facecolor='#4fc3f7', edgecolor='#4fc3f7')
ax1.add_collection3d(poly_orto)
ax1.set_title('🔲 Volumen Ortogonal\n(caja rectangular)', fontsize=11)
ax1.set_xlabel('X'); ax1.set_ylabel('Z'); ax1.set_zlabel('Y')
ax1.set_xlim(-2,2); ax1.set_ylim(0,4); ax1.set_zlim(-2,2)

# ── Frustum de Perspectiva (pirámide truncada) ────────────────
ax2 = fig.add_subplot(122, projection='3d')
ax2.set_facecolor('#1a1d27')

# En perspectiva, el plano far es más grande que el near
near_p = np.array([[-0.3,-0.3,0.5],[0.3,-0.3,0.5],[0.3,0.3,0.5],[-0.3,0.3,0.5]])
far_p  = np.array([[-1.8,-1.8,3  ],[1.8,-1.8,3  ],[1.8,1.8,3  ],[-1.8,1.8,3  ]])

caras_persp = [
    [near_p[0],near_p[1],near_p[2],near_p[3]],
    [far_p[0], far_p[1], far_p[2], far_p[3]],
    [near_p[0],near_p[1],far_p[1], far_p[0]],
    [near_p[2],near_p[3],far_p[3], far_p[2]],
    [near_p[1],near_p[2],far_p[2], far_p[1]],
    [near_p[3],near_p[0],far_p[0], far_p[3]],
]
poly_persp = Poly3DCollection(caras_persp, alpha=0.12, facecolor='#81c784', edgecolor='#81c784')
ax2.add_collection3d(poly_persp)

# Dibujamos el punto focal (vértice de la pirámide)
ax2.scatter([0],[0],[0], color='#ffcc02', s=60, zorder=5)
ax2.text(0.1, 0, 0, 'Cámara', color='#ffcc02', fontsize=9)

ax2.set_title('🔺 Frustum de Perspectiva\n(pirámide truncada)', fontsize=11)
ax2.set_xlabel('X'); ax2.set_ylabel('Z'); ax2.set_zlabel('Y')
ax2.set_xlim(-2,2); ax2.set_ylim(0,4); ax2.set_zlim(-2,2)

fig.suptitle('Volúmenes de proyección: Ortogonal vs. Perspectiva', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('volumenes_proyeccion.png', dpi=120, bbox_inches='tight')
plt.show()

print('📝 El frustum explica el efecto perspectiva:')
print('   Los objetos en el plano far ocupan más espacio en el volumen')
print('   pero se proyectan al mismo tamaño de pantalla → parecen más pequeños.')

## 📊 Sección 10 — Resumen cuantitativo

In [ ]:
import pandas as pd

# Comparamos numéricamente el efecto de distintos valores de d
# sobre las coordenadas proyectadas del vértice 8 (cumbrera del techo)
# que tiene Z=-1 (frente) y el vértice 9 que tiene Z=+1 (atrás)

print('📊 Efecto de la distancia focal sobre la proyección')
print('   Vértice analizado: 8 (cumbrera frente) vs 9 (cumbrera atrás)')
print()

filas = []
for d in [0.3, 0.5, 1.0, 2.0, 4.0, 8.0]:
    pp = proyectar_perspectiva(vertices, d=d)
    filas.append({
        'Focal d'  : d,
        'X proy v8': round(pp[0, 8], 4),
        'Y proy v8': round(pp[1, 8], 4),
        'X proy v9': round(pp[0, 9], 4),
        'Y proy v9': round(pp[1, 9], 4),
        'Tipo lente': 'Gran angular' if d < 0.8 else ('Normal' if d < 3 else 'Teleobjetivo')
    })

df = pd.DataFrame(filas)
print(df.to_string(index=False))
print()
print('✅ A mayor d, las coordenadas proyectadas convergen (menos diferencia entre v8 y v9).')